# PONITA + ECT on Kaggle (rMD17, PONITA's official training setup, 5000 epochs)

With `OFFICIAL = True` (default) this uses the same settings as `colab_official.ipynb`: PONITA's own MD17 setup (squared-error loss, per-atom summed energy, random rotations, EMA 0.99, self-loops, force-RMS scaling, 100-epoch warmup). With `OFFICIAL = False` it matches `colab_rmd17.ipynb`. Results can go in one table with the matching Colab notebook. **Run each model in only one place.**
For example, Colab runs baseline / concat / film-final and Kaggle runs one of the others.

Kaggle stops a session at 12 hours, and a 5000-epoch run needs about 25. So this notebook trains for 11 hours,
saves a checkpoint and stops cleanly. The next session picks up from that checkpoint.

### One-time setup
1. **Code dataset:** Kaggle → Datasets → New Dataset. Upload the `dino` repo folder (the same files as on Drive:
   `train_ponita_md17.py`, `models/`, `qm9_ect.py`, `ECT.py`, `EGNN.py`, `mesh_lib/`, `test_equivariance.py`).
   Don't include checkpoints or data. Name it e.g. `dino-code`.
2. **Data dataset:** a second dataset with `rmd17_aspirin.npz`. Name it e.g. `rmd17-aspirin`.
3. Create a notebook from this file (File → Import Notebook). In the right panel:
   - **Accelerator:** GPU (P100 or T4). One GPU is used.
   - **Internet:** On. This needs a phone-verified Kaggle account and is required to install PyG and PONITA.
   - **Add Input:** both datasets.
4. Set `MODEL` in Cell 2.

### Each session
- Click **Save Version → Save & Run All (Commit)**. It runs in the background, so you can close the tab.
  When it finishes, everything in `/kaggle/working/ckpts` is saved as the version's output.
- **To continue training:** open the notebook, **Add Input → Your Work → this notebook's latest version**, then commit again.
  Cell 2 copies the previous checkpoints in, and training resumes. Repeat until the log shows the `=== Summary ===` table.
- GPU time is limited to about 30 hours a week (Kaggle shows your exact quota), so plan for roughly one 5000-epoch model per week.

### Combining with Colab
Download `*_history.json` and `*_results.json` from the Kaggle output and upload them into `MyDrive/dino_ckpts_official` (or `dino_ckpts_rmd17` if `OFFICIAL = False`).
The Colab notebook's curves and results cells will then include the Kaggle models.

In [ ]:
# Cell 1 — confirm the GPU
!nvidia-smi

In [ ]:
# Cell 2 — settings, code, data and previous checkpoints
import glob, os, shutil

MODEL     = 'hybrid-adaln'   # one of: baseline, concat, film-final, leap, hybrid-adaln, hybrid-leap
EPOCHS    = 5000
SEED      = 0
MAX_HOURS = 11               # Kaggle ends a session at 12 h; stop cleanly before that
OFFICIAL  = True             # PONITA's own MD17 training setup (see top); False = earlier rMD17 setup
EXTRA     = ''               # regularised concat: MODEL = 'concat' and EXTRA = '--ect-out-dim 16 --ect-dropout 0.3'

REPO_DIR = '/tmp/dino'                 # code is copied here (inputs are read-only)
DATA_DIR = '/tmp/data'                 # PyG's processed data; rebuilt each session, not saved
CKPT_DIR = '/kaggle/working/ckpts'     # saved as the notebook output when the session ends
os.makedirs(CKPT_DIR, exist_ok=True)

import zipfile

def find(name, root='/kaggle/input'):
    hits = []
    for d, _, files in os.walk(root, followlinks=True):
        if name in files:
            hits.append(os.path.join(d, name))
    return sorted(hits)

code_files = find('train_ponita_md17.py')
if not code_files:
    # Kaggle sometimes keeps an uploaded zip as-is; unpack any zip that contains the code.
    for z in sorted(glob.glob('/kaggle/input/**/*.zip', recursive=True)):
        with zipfile.ZipFile(z) as zf:
            if any(n.endswith('train_ponita_md17.py') for n in zf.namelist()):
                zf.extractall('/tmp/code_zip')
                print('Unpacked', z)
    code_files = find('train_ponita_md17.py', '/tmp/code_zip')
if not code_files:
    print('Inputs attached to this notebook:')
    for d, dirs, files in os.walk('/kaggle/input', followlinks=True):
        depth = d[len('/kaggle/input'):].count(os.sep)
        if depth <= 3:
            print('  ' * depth + os.path.basename(d) + '/', files[:8])
    raise FileNotFoundError('train_ponita_md17.py not found in the inputs above. '
                            'Add your code dataset (Add Input), or upload a new version that contains it.')
shutil.copytree(os.path.dirname(code_files[0]), REPO_DIR, dirs_exist_ok=True)
print('Code from', os.path.dirname(code_files[0]))

# Checkpoints from a previous session of this notebook (added as an input) are copied in to resume.
for f in glob.glob('/kaggle/input/**/ckpts/*', recursive=True):
    dst = os.path.join(CKPT_DIR, os.path.basename(f))
    if os.path.isfile(f) and not os.path.exists(dst):
        shutil.copy(f, dst)
print('Checkpoints:', sorted(os.listdir(CKPT_DIR)) or 'none yet (fresh start)')

In [ ]:
# Cell 3 — install PyG and PONITA matching Kaggle's torch build
import torch
v = torch.__version__.split('+')[0]
cu = ('cu' + torch.version.cuda.replace('.', '')) if torch.version.cuda else 'cpu'
wheel = f'https://data.pyg.org/whl/torch-{v}+{cu}.html'
print('torch', torch.__version__, '->', wheel)
!pip -q install pyg-lib -f {wheel}
!pip -q install torch-scatter -f {wheel}
!pip -q install torch-geometric
!pip -q install git+https://github.com/ebekkers/ponita.git

In [ ]:
# Cell 4 — put rMD17 aspirin where PyG reads it, and build the dataset
from torch_geometric.datasets import MD17
npz = sorted(glob.glob('/kaggle/input/**/rmd17_aspirin.npz', recursive=True))
assert npz, 'Add the dataset containing rmd17_aspirin.npz as an input.'
expected = os.path.join(DATA_DIR, 'raw', 'rmd17', 'npz_data', 'rmd17_aspirin.npz')
os.makedirs(os.path.dirname(expected), exist_ok=True)
if not os.path.exists(expected):
    shutil.copy(npz[0], expected)
ds = MD17(root=DATA_DIR, name='revised aspirin')
print(f'rMD17 aspirin ready: {len(ds)} frames')

In [ ]:
# Cell 5 — sanity check: imports work and the ECT is rotation-invariant
os.chdir(REPO_DIR)
!python test_equivariance.py

In [ ]:
# Cell 6 — train (resumes from the copied checkpoints; stops cleanly after MAX_HOURS)
os.chdir(REPO_DIR)
MODEL_ARGS = '--model baseline' if MODEL == 'baseline' else f'--model ect --cond-type {MODEL}'
SETUP = '--official --warmup 100' if OFFICIAL else '--loss mse --warmup 50'
TAG   = 'official' if OFFICIAL else 'mse'
LOG = f'{CKPT_DIR}/{MODEL}{"_reg" if EXTRA else ""}_seed{SEED}_ep{EPOCHS}_{TAG}.log'
!python -u train_ponita_md17.py {MODEL_ARGS} {SETUP} \
    --molecule revised-aspirin --epochs {EPOCHS} --seed {SEED} --max-hours {MAX_HOURS} \
    --data-root "{DATA_DIR}" --ckpt-dir "{CKPT_DIR}" \
    --log-interval 25 --batch-size 5 --lr 5e-4 \
    --lambda-f 500 --hidden-dim 128 --num-layers 5 --num-ori 20 \
    --basis-dim 256 --ponita-radius 1000.0 --ect-out-dim 64 --icosphere-nu 3 {EXTRA} 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 7 — status: finished, or resume in another session?
done = glob.glob(f'{CKPT_DIR}/*_results.json')
print('FINISHED — results:', done) if done else print('Not finished yet. Commit again with this version as input to resume.')
!tail -n 5 "{LOG}"